# 05 - Final Feature Matrix Assembly

Drops raw columns that have now been superseded by engineered/encoded versions, scales numeric features (fit on training data only, needed for Logistic Regression), and saves the final modelling ready feature matrices.


## Setup

In [6]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

TRAIN_IN = Path('../../../data/processed/features_step4_train.csv')
VAL_IN = Path('../../../data/processed/features_step4_val.csv')
TEST_IN = Path('../../../data/processed/features_step4_test.csv')

TRAIN_OUT = Path('../../../data/processed/model_ready_train.csv')
VAL_OUT = Path('../../../data/processed/model_ready_val.csv')
TEST_OUT = Path('../../../data/processed/model_ready_test.csv')

train_df = pd.read_csv(TRAIN_IN)
val_df = pd.read_csv(VAL_IN)
test_df = pd.read_csv(TEST_IN)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")


Train: (46026, 38), Val: (7890, 38), Test: (11836, 38)


## 1. Drop columns superseded by engineered features

Raw columns that have been replaced by an engineered version (e.g. `Order Country`/`Order Region` replaced by their target encoded rates, `Category Name`/`Category Id` replaced by frequency encoding + the mode already baked into aggregation, `order date (DateOrders)` replaced by extracted temporal features), plus identifiers and post outcome fields that should never reach the model.


In [7]:
columns_to_drop = [
    'Order Id', 'Order Country', 'Order Region', 'Order City', 'Order State',
    'Category Name', 'Category Id', 'order date (DateOrders)',
    'Delivery Status',  
]
columns_to_drop = [c for c in columns_to_drop if c in train_df.columns]

train_df = train_df.drop(columns=columns_to_drop)
val_df = val_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

print(f"Dropped: {columns_to_drop}")
print(f"Remaining columns: {list(train_df.columns)}")


Dropped: ['Order Id', 'Order Country', 'Order Region', 'Order City', 'Order State', 'Category Name', 'Category Id', 'order date (DateOrders)', 'Delivery Status']
Remaining columns: ['Late_delivery_risk', 'Sales', 'Order Item Quantity', 'Benefit per order', 'n_line_items', 'n_distinct_categories', 'Benefit_per_order_capped', 'priority_value_component', 'order_hour', 'order_dayofweek', 'order_month', 'order_is_weekend', 'order_is_holiday_season', 'sales_per_scheduled_day', 'is_express_shipping', 'country_delay_rate', 'region_delay_rate', 'Shipping Mode_First Class', 'Shipping Mode_Same Day', 'Shipping Mode_Second Class', 'Shipping Mode_Standard Class', 'Customer Segment_Consumer', 'Customer Segment_Corporate', 'Customer Segment_Home Office', 'Type_CASH', 'Type_DEBIT', 'Type_PAYMENT', 'Type_TRANSFER', 'category_frequency']


## 2. Scale numeric features (fit on training data only)

Needed for Logistic Regression's coefficients to be comparable across features; tree-based models (Random Forest, XGBoost) don't strictly need this, but scaling doesn't hurt them either, so we apply it once for the whole feature set rather than maintaining two versions.


In [8]:
numeric_cols_to_scale = [
    'Sales', 'Order Item Quantity', 'Benefit_per_order_capped', 'n_line_items',
    'n_distinct_categories', 'Days for shipment (scheduled)', 'sales_per_scheduled_day',
    'country_delay_rate', 'region_delay_rate', 'category_frequency', 'priority_value_component',
]
numeric_cols_to_scale = [c for c in numeric_cols_to_scale if c in train_df.columns]

scaler = StandardScaler()
train_df[numeric_cols_to_scale] = scaler.fit_transform(train_df[numeric_cols_to_scale])
val_df[numeric_cols_to_scale] = scaler.transform(val_df[numeric_cols_to_scale])
test_df[numeric_cols_to_scale] = scaler.transform(test_df[numeric_cols_to_scale])

print(f"Scaled columns: {numeric_cols_to_scale}")


Scaled columns: ['Sales', 'Order Item Quantity', 'Benefit_per_order_capped', 'n_line_items', 'n_distinct_categories', 'sales_per_scheduled_day', 'country_delay_rate', 'region_delay_rate', 'category_frequency', 'priority_value_component']


**Note:** `priority_value_component` is scaled here for modelling consistency, but keep an **unscaled copy saved separately** for the actual Streamlit priority ranking display later - a scaled/standardized value (which can be negative) isn't meaningful to show an ops user as "order value."


In [9]:
priority_reference = pd.concat([
    train_df[['priority_value_component']].assign(split='train'),
    val_df[['priority_value_component']].assign(split='val'),
    test_df[['priority_value_component']].assign(split='test'),
])
unscaled_ref = pd.concat([
    pd.read_csv(Path('../../../data/processed/features_step4_train.csv'))[['priority_value_component']],
    pd.read_csv(Path('../../../data/processed/features_step4_val.csv'))[['priority_value_component']],
    pd.read_csv(Path('../../../data/processed/features_step4_test.csv'))[['priority_value_component']],
], keys=['train', 'val', 'test']).reset_index(level=0).rename(columns={'level_0': 'split'})
unscaled_ref.to_csv('../../../data/processed/priority_value_unscaled.csv', index=False)
print("Saved unscaled priority value reference for later app use.")


Saved unscaled priority value reference for later app use.


## 3. Final checks and save

In [10]:
assert train_df.isnull().sum().sum() == 0, "Missing values found in final train set!"
assert val_df.isnull().sum().sum() == 0, "Missing values found in final val set!"
assert test_df.isnull().sum().sum() == 0, "Missing values found in final test set!"

print(f"Final shapes — Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
print(f"\nFinal feature columns (excluding target):")
print([c for c in train_df.columns if c != 'Late_delivery_risk'])

train_df.to_csv(TRAIN_OUT, index=False)
val_df.to_csv(VAL_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)
print("\nSaved model-ready train/val/test sets.")


Final shapes — Train: (46026, 29), Val: (7890, 29), Test: (11836, 29)

Final feature columns (excluding target):
['Sales', 'Order Item Quantity', 'Benefit per order', 'n_line_items', 'n_distinct_categories', 'Benefit_per_order_capped', 'priority_value_component', 'order_hour', 'order_dayofweek', 'order_month', 'order_is_weekend', 'order_is_holiday_season', 'sales_per_scheduled_day', 'is_express_shipping', 'country_delay_rate', 'region_delay_rate', 'Shipping Mode_First Class', 'Shipping Mode_Same Day', 'Shipping Mode_Second Class', 'Shipping Mode_Standard Class', 'Customer Segment_Consumer', 'Customer Segment_Corporate', 'Customer Segment_Home Office', 'Type_CASH', 'Type_DEBIT', 'Type_PAYMENT', 'Type_TRANSFER', 'category_frequency']

Saved model-ready train/val/test sets.


**What we found:**

**Final shapes: Train (46,026, 29), Validation (7,890, 29), Test (11,836, 29)** - matches expectations (38 columns after Notebook 04's encoding, minus 9 dropped raw/redundant columns = 29). No missing values in any split (all three assertions passed). The 27 feature final set (28 columns minus the target) spans: order value/quantity features, the capped profit and priority value fields, five temporal features, the two shipping context features from Notebook 02, two geographic target encoded rates, 11 one hot encoded columns across Shipping Mode/Customer Segment/Type, and the category frequency encoding.

**Stage 4 is complete.** The model ready train/validation/test sets are saved and ready for Stage 5/6 (baseline and model development).

`DECISION_LOG.md`:** final feature set locked in for modelling, list of dropped raw columns, scaling approach (StandardScaler fit on train only), and the decision to keep an unscaled priority-value reference table separate from the modelling matrix for later app use. Also worth a short note that Notebook 02 required a correction (building `sales_per_scheduled_day` before dropping its source column, not after) - a small but genuine example of catching and fixing a pipeline ordering bug before it reached modelling.

